# Quantum chemistry basics with PySCF

PySCF solves the electronic structure problem: given a molecule's atoms and geometry, find
the electrons' wavefunction and the resulting energy. We compute the Hartree-Fock energy of
H\(_2\) (the simplest possible molecule) at its known experimental bond length, then scan
over bond length to confirm the energy has a minimum where chemistry says it should.

In [ ]:
from pyscf import gto, scf

# H2 at its experimental equilibrium bond length (0.74 Angstrom), a minimal STO-3G basis set
mol = gto.M(atom="H 0 0 0; H 0 0 0.74", basis="sto-3g", verbose=0)
mf = scf.RHF(mol)
energy = mf.kernel()
print(f"H2 RHF/STO-3G energy at r=0.74A: {energy:.6f} Hartree")
print("(textbook reference value: approximately -1.117 Hartree)")

## Does the energy actually have a minimum where it should?

Real chemistry: pull the two hydrogen atoms too close together and they repel (nuclear
repulsion dominates); pull them too far apart and the bond breaks. Somewhere in between is the
stable bond length. We scan bond length and check the computed minimum lines up with the real,
experimentally measured H\(_2\) bond length of 0.74 A.

In [ ]:
import numpy as np

bond_lengths = np.linspace(0.5, 1.5, 21)
energies = []
for r in bond_lengths:
    mol = gto.M(atom=f"H 0 0 0; H 0 0 {r}", basis="sto-3g", verbose=0)
    energies.append(scf.RHF(mol).kernel())
energies = np.array(energies)

r_min = bond_lengths[np.argmin(energies)]
print(f"computed energy minimum at r = {r_min:.2f} A")
print(f"experimental equilibrium bond length: 0.74 A")
print(f"E(minimum) = {energies.min():.6f} Hartree")
print()
print("Note: STO-3G is a very small/minimal basis set, so a small discrepancy from 0.74 A")
print("is expected and normal, not a bug - STO-3G is known to systematically underestimate")
print("bond lengths. Try basis='6-31g' or basis='cc-pvdz' below and see the minimum move")
print("closer to 0.74 A as the basis set improves.")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(7, 4))
plt.plot(bond_lengths, energies, "o-")
plt.axvline(0.74, color="tab:red", linestyle="--", label="experimental (0.74 A)")
plt.axvline(r_min, color="tab:green", linestyle="--", label=f"computed minimum ({r_min:.2f} A)")
plt.xlabel("H-H bond length (Angstrom)")
plt.ylabel("energy (Hartree)")
plt.title("H2 potential energy curve (RHF/STO-3G)")
plt.legend()
plt.show()

## Next steps

- Swap `basis="sto-3g"` for a larger basis set (`"6-31g"`, `"cc-pvdz"`) and watch both the
  energy and the computed equilibrium bond length improve toward experiment.
- Try a bigger molecule (e.g. water: `"O 0 0 0; H 0 0.76 -0.48; H 0 -0.76 -0.48"`).
- Go beyond Hartree-Fock: `from pyscf import cc` and `cc.CCSD(mf).kernel()` adds electron
  correlation for a more accurate (and much more expensive) energy.